# Auditoria y consolidacion del test - Objetivo 4
Semillas 42, 123 y 2026; DenseNet121 frente a CNN-ViT.

Ejecuta las celdas en orden. No requiere GPU, checkpoints ni el dataset de imagenes.
Solo lee resultados existentes; no entrena ni selecciona umbrales. Guarda una carpeta nueva
en `cxr-thesis/audits`. No uses Ejecutar todo en el cuaderno de entrenamiento.

Esta es una evaluacion interna retrospectiva: el test ya fue inspeccionado.
Las semillas no son cohortes independientes. La desviacion entre semillas no es un IC.
El bootstrap puede tardar; muestra progreso cada 50 replicas.


In [1]:
%pip -q install numpy pandas scikit-learn


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys, zipfile
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display

DRIVE_ROOT = Path('/content/drive/MyDrive/cxr-thesis')
EXPERIMENTS = DRIVE_ROOT / 'experiments/publication_comparison_v1'
SEEDS = (42, 123, 2026)
MODELS = ('densenet121', 'cnn_vit')
TOLERANCE = 1e-8
RUN_BOOTSTRAP = True
# El comparador original usa seed=42 para TODAS las semillas de entrenamiento.
BOOTSTRAP_SEED = 42
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f_UTC')
OUTPUT = DRIVE_ROOT / 'audits' / ('test_tres_semillas_' + timestamp)
if not EXPERIMENTS.is_dir():
    raise FileNotFoundError(f'No se encuentra {EXPERIMENTS}. Revisa cuenta y montaje de Drive.')
print('Origen:', EXPERIMENTS)
print('Salida nueva:', OUTPUT)


Mounted at /content/drive
Origen: /content/drive/MyDrive/cxr-thesis/experiments/publication_comparison_v1
Salida nueva: /content/drive/MyDrive/cxr-thesis/audits/test_tres_semillas_20261008_185345_898191_UTC


## 1. Funciones de verificacion
El bootstrap reproduce el algoritmo del comparador original: muestrea pacientes con reemplazo, conserva todas sus imagenes y calcula diferencias candidato menos baseline. No combina pacientes de distintas semillas.


In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def assert_close(a, b, context):
    if not np.allclose(a, b, atol=TOLERANCE, rtol=0):
        raise ValueError(f'No coincide {context}: calculado={a}, archivado={b}')

def load_arrays(path, classes):
    with np.load(path, allow_pickle=False) as archive:
        arrays = {key: archive[key].copy() for key in ('labels', 'scores', 'patients')}
    y, s, p = (arrays[key] for key in ('labels', 'scores', 'patients'))
    if y.ndim != 2 or y.shape != s.shape or y.shape[1] != len(classes):
        raise ValueError(f'Forma incorrecta: {path}')
    if p.ndim != 1 or len(p) != len(y) or not len(y):
        raise ValueError(f'Pacientes incorrectos: {path}')
    if not np.isfinite(y).all() or not np.isfinite(s).all():
        raise ValueError(f'Valores no finitos: {path}')
    if not np.isin(y, [0, 1]).all() or np.any((s < 0) | (s > 1)):
        raise ValueError(f'Etiquetas o scores invalidos: {path}')
    if any(np.unique(y[:, i]).size != 2 for i in range(y.shape[1])):
        raise ValueError(f'Clase sin positivos o negativos: {path}')
    return arrays


In [4]:
def class_metrics(labels: np.ndarray, scores: np.ndarray, thresholds: np.ndarray) -> dict:
    report = {}
    for index, threshold in enumerate(thresholds):
        truth = labels[:, index].astype(bool)
        pred = scores[:, index] >= threshold
        tp = int(np.logical_and(truth, pred).sum())
        fp = int(np.logical_and(~truth, pred).sum())
        fn = int(np.logical_and(truth, ~pred).sum())
        tn = int(np.logical_and(~truth, ~pred).sum())
        report[index] = {
            "auc": float(roc_auc_score(truth, scores[:, index])),
            "ap": float(average_precision_score(truth, scores[:, index])),
            "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1),
            "precision": tp / max(tp + fp, 1),
            "npv": tn / max(tn + fn, 1),
            "f1": 2 * tp / max(2 * tp + fp + fn, 1),
            "threshold": float(threshold),
            "positives": int(truth.sum()),
        }
    return report


In [5]:
def paired_bootstrap(labels, baseline, candidate, patients, replicates=500, seed=42):
    if labels.shape != baseline.shape or labels.shape != candidate.shape or len(labels) != len(patients):
        raise ValueError("Prediction/label/patient shapes differ")
    if replicates < 1 or not all(np.isfinite(a).all() for a in (labels, baseline, candidate)):
        raise ValueError("Invalid replicates or non-finite arrays")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must be binary")
    groups = [np.flatnonzero(patients == patient) for patient in np.unique(patients)]
    generator = np.random.default_rng(seed)

    def deltas(rows):
        auc, ap = [], []
        for i in range(labels.shape[1]):
            y = labels[rows, i]
            auc.append(roc_auc_score(y, candidate[rows, i]) - roc_auc_score(y, baseline[rows, i]))
            ap.append(average_precision_score(y, candidate[rows, i]) - average_precision_score(y, baseline[rows, i]))
        return float(np.mean(auc)), float(np.mean(ap))

    if any(np.unique(labels[:, i]).size < 2 for i in range(labels.shape[1])):
        raise ValueError("Each class requires positive and negative test examples")
    point = deltas(np.arange(len(labels)))
    results = []
    for iteration in range(replicates):
        if iteration % 50 == 0:
            print(f"Bootstrap {iteration}/{replicates}", flush=True)
        rows = np.concatenate([groups[i] for i in generator.integers(0, len(groups), len(groups))])
        if any(np.unique(labels[rows, i]).size < 2 for i in range(labels.shape[1])):
            continue
        results.append(deltas(rows))
    if not results:
        raise ValueError("No valid bootstrap replicates; too few independent patients")
    intervals = np.quantile(np.asarray(results), [0.025, 0.975], axis=0)
    return {"delta_definition": "candidate minus baseline", "unit": "patient", "replicates": len(results),
            "requested_replicates": replicates,
            "auc_macro_delta": point[0], "auc_macro_delta_95ci": intervals[:, 0].tolist(),
            "map_delta": point[1], "map_delta_95ci": intervals[:, 1].tolist()}


## 2. Auditoria de los seis resultados
Si encuentra un archivo faltante, cambio de hash, protocolo diferente o metricas inconsistentes, se detiene. No corrige resultados silenciosamente. Tambien compara validacion y test cuando las predicciones de validacion estan disponibles.


In [6]:
records, macro_rows, class_rows, inventory = {}, [], [], []
protocol_reference, test_reference = None, None
METRICS = ('auc', 'ap', 'sensitivity', 'specificity', 'precision', 'npv', 'f1')
PROTOCOL_KEYS = ('hdf5_sha256', 'split_sha256', 'classes', 'preprocessing',
                 'initialization', 'code_sha256', 'epochs', 'warmup', 'patience',
                 'batch_size', 'accumulation', 'cnn_lr', 'head_lr', 'loss')
for seed in SEEDS:
    for model in MODELS:
        folder = EXPERIMENTS / f'imagenet-bce-seed{seed}' / model
        required = ('test_report.json', 'test_predictions.npz', 'config.json',
                    'thresholds.json', 'training_summary.json')
        for name in required:
            path = folder / name
            if not path.is_file():
                raise FileNotFoundError(f'Falta {path}')
            inventory.append({'path': str(path.relative_to(DRIVE_ROOT)),
                              'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
        report, config = read_json(folder / required[0]), read_json(folder / 'config.json')
        summary = read_json(folder / 'training_summary.json')
        thresholds = read_json(folder / 'thresholds.json')
        if report['status'] != 'completed' or report['config'] != config or summary['config'] != config:
            raise ValueError(f'Estado/configuracion inconsistente: {folder}')
        if config['seed'] != seed or config['architecture'] != model or config['loss'] != 'bce':
            raise ValueError(f'Identidad de corrida inconsistente: {folder}')
        if report.get('test_previously_inspected') is not True:
            raise ValueError('Revisar declaracion de inspeccion previa de test.')
        if sha256_file(folder / 'test_predictions.npz') != report['prediction_sha256']:
            raise ValueError(f'Hash de predicciones incorrecto: {folder}')
        if sha256_file(folder / 'thresholds.json') != summary['thresholds_sha256']:
            raise ValueError(f'Hash de umbrales incorrecto: {folder}')
        if thresholds != report['thresholds']:
            raise ValueError(f'Umbrales diferentes: {folder}')
        if report['best_checkpoint_sha256'] != summary['best_checkpoint_sha256']:
            raise ValueError(f'Referencia de checkpoint diferente: {folder}')
        protocol = {key: config[key] for key in PROTOCOL_KEYS}
        if protocol_reference is None:
            protocol_reference = protocol
        elif protocol != protocol_reference:
            raise ValueError(f'Protocolo diferente: {folder}')
        classes = config['classes']
        if len(classes) != 14 or len(set(classes)) != 14:
            raise ValueError('Orden/lista de clases inesperado.')
        arrays = load_arrays(folder / 'test_predictions.npz', classes)
        y, scores, patients = (arrays[key] for key in ('labels', 'scores', 'patients'))
        if test_reference is None:
            test_reference = (y, patients)
        elif not np.array_equal(y, test_reference[0]) or not np.array_equal(patients, test_reference[1]):
            raise ValueError(f'Etiquetas u orden de pacientes diferentes: {folder}')
        t = np.asarray([thresholds[name] for name in classes], dtype=float)
        if not np.isfinite(t).all() or np.any((t < 0) | (t > 1)):
            raise ValueError('Umbrales invalidos.')
        calculated = class_metrics(y, scores, t)
        macro = {key: float(np.mean([row[key] for row in calculated.values()])) for key in METRICS}
        for key in METRICS:
            assert_close(macro[key], report['test_metrics']['macro'][key], f'{seed}/{model}/macro/{key}')
        for i, name in enumerate(classes):
            for key, value in calculated[i].items():
                assert_close(value, report['test_metrics'][name][key], f'{seed}/{model}/{name}/{key}')
            class_rows.append({'seed': seed, 'model': model, 'class': name, **calculated[i]})
        validation_path = folder / 'validation_predictions.npz'
        validation_checked = False
        if validation_path.exists():
            if sha256_file(validation_path) != summary['validation_predictions_sha256']:
                raise ValueError(f'Hash de validacion incorrecto: {folder}')
            validation = load_arrays(validation_path, classes)
            if np.intersect1d(validation['patients'], patients).size:
                raise ValueError(f'Pacientes compartidos entre validacion y test: {folder}')
            validation_checked = True
        macro_rows.append({'seed': seed, 'model': model, 'images': len(y),
                           'patients': len(np.unique(patients)), **macro,
                           'validation_test_disjoint_checked': validation_checked})
        records[(seed, model)] = {'arrays': arrays, 'report': report, 'config': config}
        print(f'OK: {seed} / {model}; {len(y)} imagenes; {len(np.unique(patients))} pacientes')
macro_table = pd.DataFrame(macro_rows)
class_table = pd.DataFrame(class_rows)
display(macro_table)
print('Metricas verificadas. Checkpoints, dataset y separacion con train NO verificados en esta auditoria.')


OK: 42 / densenet121; 16578 imagenes; 4621 pacientes
OK: 42 / cnn_vit; 16578 imagenes; 4621 pacientes
OK: 123 / densenet121; 16578 imagenes; 4621 pacientes
OK: 123 / cnn_vit; 16578 imagenes; 4621 pacientes
OK: 2026 / densenet121; 16578 imagenes; 4621 pacientes
OK: 2026 / cnn_vit; 16578 imagenes; 4621 pacientes


,seed,model,images,patients,auc,ap,sensitivity,specificity,precision,npv,f1,validation_test_disjoint_checked
0,42,densenet121,16578,4621,0.818400,0.244961,0.724538,0.766388,0.136987,0.978882,0.216374,True
1,42,cnn_vit,16578,4621,0.817636,0.246325,0.740844,0.746762,0.129868,0.978772,0.204010,True
2,123,densenet121,16578,4621,0.819052,0.246346,0.720257,0.770804,0.136006,0.979029,0.216459,True
3,123,cnn_vit,16578,4621,0.815694,0.244615,0.716567,0.771337,0.136941,0.977842,0.214863,True
4,2026,densenet121,16578,4621,0.817770,0.241715,0.730525,0.762964,0.133807,0.978815,0.212886,True
5,2026,cnn_vit,16578,4621,0.816520,0.247406,0.746398,0.741149,0.133297,0.978592,0.208416,True


Metricas verificadas. Checkpoints, dataset y separacion con train NO verificados en esta auditoria.


## 3. Reproduccion de las comparaciones emparejadas
Se conserva el numero de replicas del archivo original y la semilla aleatoria 42 del comparador. Los intervalos archivados y reproducidos se muestran por separado. Una discrepancia se registra como fallo; no se reemplaza el original.


In [7]:
paired_rows, comparisons = [], {}
for seed in SEEDS:
    path = EXPERIMENTS / f'imagenet-bce-seed{seed}' / 'paired_comparison.json'
    archived = read_json(path)
    inventory.append({'path': str(path.relative_to(DRIVE_ROOT)),
                      'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
    b, c = records[(seed, 'densenet121')], records[(seed, 'cnn_vit')]
    if archived['baseline_config'] != b['config'] or archived['candidate_config'] != c['config']:
        raise ValueError(f'Configuracion de comparacion incorrecta: {seed}')
    if archived['unit'] != 'patient' or archived.get('test_previously_inspected') is not True:
        raise ValueError(f'Declaracion de comparacion incorrecta: {seed}')
    assert_close(c['report']['test_metrics']['macro']['auc'] - b['report']['test_metrics']['macro']['auc'],
                 archived['auc_macro_delta'], f'delta AUROC {seed}')
    assert_close(c['report']['test_metrics']['macro']['ap'] - b['report']['test_metrics']['macro']['ap'],
                 archived['map_delta'], f'delta mAP {seed}')
    reproduced, matches = None, None
    if RUN_BOOTSTRAP:
        print(f'Iniciando comparacion de semilla {seed}', flush=True)
        reproduced = paired_bootstrap(b['arrays']['labels'], b['arrays']['scores'],
                                      c['arrays']['scores'], b['arrays']['patients'],
                                      replicates=int(archived['requested_replicates']), seed=BOOTSTRAP_SEED)
        keys = ('auc_macro_delta', 'auc_macro_delta_95ci', 'map_delta', 'map_delta_95ci')
        matches = all(np.allclose(reproduced[k], archived[k], atol=TOLERANCE, rtol=0) for k in keys)
        matches = matches and reproduced['replicates'] == archived['replicates']
    comparisons[str(seed)] = {'archived': archived, 'reproduced': reproduced,
                              'matches_archive': matches}
    paired_rows.append({'seed': seed, 'auc_delta': archived['auc_macro_delta'],
                        'auc_ci_low': archived['auc_macro_delta_95ci'][0],
                        'auc_ci_high': archived['auc_macro_delta_95ci'][1],
                        'map_delta': archived['map_delta'],
                        'map_ci_low': archived['map_delta_95ci'][0],
                        'map_ci_high': archived['map_delta_95ci'][1],
                        'bootstrap_reproduced_matches': matches})
    print('Coincide con archivo:', matches)
paired_table = pd.DataFrame(paired_rows)
display(paired_table)


Iniciando comparacion de semilla 42
Bootstrap 0/500
Bootstrap 50/500
Bootstrap 100/500
Bootstrap 150/500
Bootstrap 200/500
Bootstrap 250/500
Bootstrap 300/500
Bootstrap 350/500
Bootstrap 400/500
Bootstrap 450/500
Coincide con archivo: True
Iniciando comparacion de semilla 123
Bootstrap 0/500
Bootstrap 50/500
Bootstrap 100/500
Bootstrap 150/500
Bootstrap 200/500
Bootstrap 250/500
Bootstrap 300/500
Bootstrap 350/500
Bootstrap 400/500
Bootstrap 450/500
Coincide con archivo: True
Iniciando comparacion de semilla 2026
Bootstrap 0/500
Bootstrap 50/500
Bootstrap 100/500
Bootstrap 150/500
Bootstrap 200/500
Bootstrap 250/500
Bootstrap 300/500
Bootstrap 350/500
Bootstrap 400/500
Bootstrap 450/500
Coincide con archivo: True


,seed,auc_delta,auc_ci_low,auc_ci_high,map_delta,map_ci_low,map_ci_high,bootstrap_reproduced_matches
0,42,-0.000764,-0.004455,0.003843,0.001365,-0.007792,0.009070,True
1,123,-0.003359,-0.008163,0.002700,-0.001730,-0.008372,0.005350,True
2,2026,-0.001249,-0.005199,0.003253,0.005691,-0.000674,0.012956,True


## 4. Consolidacion y exportacion
La media y la desviacion estandar describen tres entrenamientos sobre la misma particion. No promediar limites de IC ni presentar estas tres semillas como tres cohortes. Se exportan agregados, no predicciones ni identificadores de pacientes.


In [8]:
summary_table = macro_table.groupby('model')[list(METRICS)].agg(['mean', 'std'])
summary_table.columns = ['_'.join(column) for column in summary_table.columns]
display(summary_table)
bootstrap_ok = RUN_BOOTSTRAP and all(value['matches_archive'] is True for value in comparisons.values())
status = 'metrics_and_paired_bootstrap_verified' if bootstrap_ok else 'metrics_verified_bootstrap_pending_or_mismatch'
OUTPUT.mkdir(parents=True, exist_ok=False)
macro_table.to_csv(OUTPUT / 'test_metrics_six_runs.csv', index=False)
class_table.to_csv(OUTPUT / 'test_metrics_per_class.csv', index=False)
summary_table.to_csv(OUTPUT / 'test_summary_mean_sd.csv')
paired_table.to_csv(OUTPUT / 'paired_comparisons_three_seeds.csv', index=False)
audit = {'status': status, 'created_utc': timestamp, 'seeds': list(SEEDS),
         'metrics_tolerance': TOLERANCE, 'bootstrap_seed': BOOTSTRAP_SEED,
         'source_files': inventory, 'comparisons': comparisons,
         'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                         'pandas': pd.__version__, 'sklearn': sklearn.__version__},
         'limits': ['Retrospective internal test, previously inspected',
                    'No external or clinical validation',
                    'Training seeds are not independent patient cohorts',
                    'Checkpoint binaries, image dataset and train overlap not verified',
                    'Individual-model bootstrap intervals not regenerated; paired intervals only']}
(OUTPUT / 'audit_report.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
interpretation = (
    '# Auditoria de test de tres semillas\n\n'
    f'Estado: {status}\n\n'
    'Se recalcularon las siete metricas macro y las metricas por clase desde las seis predicciones.\n'
    'Los intervalos emparejados son candidato menos baseline y muestrean pacientes.\n'
    'Consultar audit_report.json para replicas, diferencias y coincidencia con archivos originales.\n'
    'La media y SD son descriptivas; no son una prueba de superioridad o equivalencia.\n'
    'La auditoria no cierra las pruebas funcionales, de calidad o Grad-CAM del Objetivo 4.\n'
)
(OUTPUT / 'LEEME.md').write_text(interpretation, encoding='utf-8')
zip_path = OUTPUT.parent / (OUTPUT.name + '.zip')
with zipfile.ZipFile(zip_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.iterdir()):
        archive.write(path, arcname=path.name)
print('Auditoria guardada:', OUTPUT)
print('ZIP:', zip_path)
if not bootstrap_ok:
    print('ATENCION: bootstrap no confirmado. Enviame el ZIP antes de usar esos intervalos como reproducidos.')
from google.colab import files
files.download(str(zip_path))


,auc_mean,auc_std,ap_mean,ap_std,sensitivity_mean,sensitivity_std,specificity_mean,specificity_std,precision_mean,precision_std,npv_mean,npv_std,f1_mean,f1_std
model,,,,,,,,,,,,,,
cnn_vit,0.816617,0.000975,0.246116,0.001407,0.734603,0.015865,0.753083,0.016056,0.133369,0.003537,0.978402,0.000493,0.209096,0.005459
densenet121,0.818407,0.000641,0.244341,0.002377,0.725107,0.005158,0.766719,0.003930,0.135600,0.001628,0.978909,0.000110,0.215240,0.002039


Auditoria guardada: /content/drive/MyDrive/cxr-thesis/audits/test_tres_semillas_20261008_185345_898191_UTC
ZIP: /content/drive/MyDrive/cxr-thesis/audits/test_tres_semillas_20261008_185345_898191_UTC.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>